# CGS410 — Embedding Extraction (mBERT)
**Do Case Markers Reduce Contextual Encoding of Syntactic Role in LLMs?**

Archit Rahalkar (240172) · Taran Mohta (241095)

---

**Inputs:** `outputs/en_filtered.csv`, `outputs/hi_filtered.csv`, `outputs/tr_filtered.csv` (from `01_data_prep.ipynb`)

**Outputs:**
- `outputs/en_embeddings.npz` — shape `(N_en, 13, 768)` — all 13 mBERT layers for each English token
- `outputs/hi_embeddings.npz` — shape `(N_hi, 13, 768)` — same for Hindi
- `outputs/tr_embeddings.npz` — shape `(N_tr, 13, 768)` — same for Turkish
- `outputs/en_meta.csv` / `outputs/hi_meta.csv` / `outputs/tr_meta.csv` — row-aligned metadata (lemma, role, sent_id …)

Layer 0 = embedding layer, layers 1–12 = transformer layers.


## 0 · Install

In [ ]:
!pip install -q transformers torch pandas numpy tqdm

## 1 · Config

In [ ]:
from pathlib import Path
import torch

OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)

MBERT_NAME  = "bert-base-multilingual-cased"
MAX_SEQ_LEN = 128
BATCH_SIZE  = 16

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")

Device: cpu


## 2 · Load filtered data

In [2]:
import pandas as pd

en_df = pd.read_csv(OUT_DIR / "en_balanced.csv")
hi_df = pd.read_csv(OUT_DIR / "hi_balanced.csv")
tr_df = pd.read_csv(OUT_DIR / "tr_balanced.csv")

for lang, df in [("English", en_df), ("Hindi", hi_df), ("Turkish", tr_df)]:
    print(f"[{lang}] {len(df):,} tokens | {df['role'].value_counts().to_dict()}")


[English] 3,434 tokens | {'subject': 1717, 'object': 1717}
[Hindi] 3,434 tokens | {'subject': 1717, 'object': 1717}
[Turkish] 3,434 tokens | {'subject': 1717, 'object': 1717}


## 3 · Load mBERT

In [ ]:
from transformers import BertTokenizerFast, BertModel

print(f"Loading {MBERT_NAME} ...")
tokenizer = BertTokenizerFast.from_pretrained(MBERT_NAME)
model     = BertModel.from_pretrained(MBERT_NAME, output_hidden_states=True)
model.eval()
model.to(DEVICE)
print(f"Model loaded. Parameters: {sum(p.numel() for p in model.parameters()):,}")

## 4 · Token alignment helper

mBERT uses WordPiece tokenization, so a single CoNLL word may map to multiple sub-tokens.
We use the tokenizer's **word_ids** (available with `BertTokenizerFast`) to find all
sub-token positions that belong to the target word, then **mean-pool** them into one vector.

The CoNLL `token_idx` is 1-based; `word_ids` are 0-based and `None` for special tokens (`[CLS]`, `[SEP]`).

In [ ]:
import numpy as np

def get_word_subtoken_positions(word_ids: list, target_word_idx: int) -> list[int]:
    return [pos for pos, wid in enumerate(word_ids) if wid == target_word_idx]


def extract_token_embedding(hidden_states, positions: list[int]) -> np.ndarray:
    layers = []
    for layer_hidden in hidden_states:
        subtoken_vecs = layer_hidden[positions]
        layers.append(subtoken_vecs.mean(dim=0).cpu().numpy())
    return np.stack(layers)


print("Helpers defined.")

Helpers defined.


## 5 · Batch embedding extraction

We batch sentences together for efficiency. For each sentence in a batch we:
1. Tokenize with `BertTokenizerFast` (keeping word-level alignment via `word_ids`)
2. Forward pass through mBERT with `output_hidden_states=True`
3. For each token of interest in that sentence, find its sub-token positions and mean-pool across all 13 layers

In [ ]:
from tqdm.auto import tqdm

def extract_embeddings(df: pd.DataFrame) -> tuple[np.ndarray, pd.DataFrame]:
    grouped = df.groupby("sent_id", sort=False)

    sentences   = []
    sent_targets = []

    for sent_id, group in grouped:
        sentence_tokens = str(group.iloc[0]["sentence"]).split()
        targets = [
            (row_idx, int(row["token_idx"]) - 1)
            for row_idx, row in group.iterrows()
        ]
        sentences.append(sentence_tokens)
        sent_targets.append(targets)

    N = len(df)
    all_embeddings = np.zeros((N, 13, 768), dtype=np.float32)
    rows_filled = set()

    n_batches = (len(sentences) + BATCH_SIZE - 1) // BATCH_SIZE

    for batch_start in tqdm(range(0, len(sentences), BATCH_SIZE),
                            total=n_batches, desc="Batches"):
        batch_sents   = sentences[batch_start : batch_start + BATCH_SIZE]
        batch_targets = sent_targets[batch_start : batch_start + BATCH_SIZE]

        encoding = tokenizer(
            batch_sents,
            is_split_into_words=True,
            padding=True,
            truncation=True,
            max_length=MAX_SEQ_LEN,
            return_tensors="pt",
            return_offsets_mapping=False,
        )
        input_ids      = encoding["input_ids"].to(DEVICE)
        attention_mask = encoding["attention_mask"].to(DEVICE)

        batch_word_ids = [encoding.word_ids(batch_index=i)
                          for i in range(len(batch_sents))]

        with torch.no_grad():
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
            )
        hidden_states = outputs.hidden_states

        for sent_i, targets in enumerate(batch_targets):
            word_ids = batch_word_ids[sent_i]
            sent_hidden = tuple(layer[sent_i] for layer in hidden_states)

            for df_row_idx, word_idx_0based in targets:
                positions = get_word_subtoken_positions(word_ids, word_idx_0based)

                if len(positions) == 0:
                    preview = " ".join(batch_sents[sent_i][:20])
                    print(f"  WARNING: word truncated at row {df_row_idx} "
                          f"(word_idx={word_idx_0based}, sent='{preview} ...')")
                    continue

                row = df.loc[df_row_idx]
                out_idx = df.index.get_loc(df_row_idx)
                all_embeddings[out_idx] = extract_token_embedding(sent_hidden, positions)
                rows_filled.add(out_idx)

    meta = df[["sent_id", "token_idx", "form", "lemma", "role", "deprel", "split"]].reset_index(drop=True)
    
    if len(rows_filled) < N:
        n_truncated = N - len(rows_filled)
        print(f"  Filtered out {n_truncated} truncated token(s).")
        filled_idx = np.array(sorted(rows_filled), dtype=int)
        all_embeddings = all_embeddings[filled_idx]
        meta = meta.loc[filled_idx].reset_index(drop=True)

    return all_embeddings, meta

In [6]:
print("Extracting English embeddings ...")
en_embeddings, en_meta = extract_embeddings(en_df)
print(f"  Shape: {en_embeddings.shape}")

print("\nExtracting Hindi embeddings ...")
hi_embeddings, hi_meta = extract_embeddings(hi_df)
print(f"  Shape: {hi_embeddings.shape}")

print("\nExtracting Turkish embeddings ...")
tr_embeddings, tr_meta = extract_embeddings(tr_df)
print(f"  Shape: {tr_embeddings.shape}")


Extracting English embeddings ...


Batches:  72%|███████▏  | 137/190 [02:35<00:48,  1.10it/s]

Batches: 100%|██████████| 190/190 [03:15<00:00,  1.03s/it]


  Filtered out 1 truncated token(s).
  Shape: (3433, 13, 768)

Extracting Hindi embeddings ...


Batches:  42%|████▏     | 82/196 [01:23<02:15,  1.19s/it]

Batches: 100%|██████████| 196/196 [03:31<00:00,  1.08s/it]


  Filtered out 1 truncated token(s).
  Shape: (3433, 13, 768)

Extracting Turkish embeddings ...


Batches: 100%|██████████| 160/160 [02:32<00:00,  1.05it/s]

  Shape: (3434, 13, 768)


## 7 · Save

In [ ]:
np.savez_compressed(OUT_DIR / "en_embeddings.npz", embeddings=en_embeddings)
np.savez_compressed(OUT_DIR / "hi_embeddings.npz", embeddings=hi_embeddings)
np.savez_compressed(OUT_DIR / "tr_embeddings.npz", embeddings=tr_embeddings)

en_meta.to_csv(OUT_DIR / "en_meta.csv", index=False)
hi_meta.to_csv(OUT_DIR / "hi_meta.csv", index=False)
tr_meta.to_csv(OUT_DIR / "tr_meta.csv", index=False)

print("Saved to outputs/:")
for f in sorted(OUT_DIR.iterdir()):
    print(f"  {f.name:<30} {f.stat().st_size / (1024**2):.1f} MB")

Saved to outputs/:
  cosine_distance_by_layer.csv   0.0 MB
  en_balanced.csv                0.7 MB
  en_embeddings.npz              121.3 MB
  en_filtered.csv                2.6 MB
  en_lemma_dist_layer8.csv       0.0 MB
  en_meta.csv                    0.3 MB
  en_tokens.csv                  3.5 MB
  hi_balanced.csv                1.3 MB
  hi_embeddings.npz              121.3 MB
  hi_filtered.csv                8.9 MB
  hi_lemma_dist_layer8.csv       0.0 MB
  hi_meta.csv                    0.2 MB
  hi_tokens.csv                  10.9 MB
  probe_accuracy_by_layer.csv    0.0 MB
  tr_balanced.csv                0.5 MB
  tr_embeddings.npz              121.3 MB
  tr_filtered.csv                0.6 MB
  tr_meta.csv                    0.2 MB
  tr_tokens.csv                  0.9 MB
